In [1]:
import joblib
import numpy as np
import streamlit as st
import pandas as pd
import re

# If you already have clean_tweet in another file, import it instead
# from src.preprocess import clean_tweet

def clean_tweet(text):
    text = str(text).lower().strip()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

@st.cache_resource
def load_artifacts():
    bundle = joblib.load("twitter_sentiment_model_artifacts.joblib")
    return bundle["model_pipeline"], bundle["label_encoder"]

model, lbl_enc = load_artifacts()

st.set_page_config(page_title="Twitter Sentiment Analyzer", page_icon="💬")
st.title("💬 Twitter Sentiment Analyzer")
st.write("Enter text to predict sentiment using a trained TF-IDF + ML model.")

user_text = st.text_area("Enter text", height=120, placeholder="Type a tweet or message...")

if st.button("Predict Sentiment"):
    if not user_text.strip():
        st.warning("Please enter some text.")
    else:
        cleaned = clean_tweet(user_text)
        pred_id = model.predict([cleaned])[0]
        pred_label = lbl_enc.inverse_transform([pred_id])[0]

        st.subheader("Prediction")
        st.success(f"**Sentiment:** {pred_label}")

        if hasattr(model, "predict_proba"):
            probs = model.predict_proba([cleaned])[0]
            confidence = float(np.max(probs))
            st.write(f"**Confidence:** {confidence:.2%}")

            prob_df = pd.DataFrame({
                "Sentiment": lbl_enc.inverse_transform(np.arange(len(probs))),
                "Probability": probs
            }).sort_values("Probability", ascending=False)

            st.subheader("Class Probabilities")
            st.dataframe(prob_df, use_container_width=True)

        with st.expander("Show cleaned text"):
            st.code(cleaned)

ModuleNotFoundError: No module named 'streamlit'